# HyDE

## 💭 What is HyDe?

HyDE (**Hypothetical Document Embeddings**) is a retrieval technique where, instead of embedding the **user's query** directly, you first generate a **hypothetical answer (document)** to the query using an **LLM** — and then embed that hypothetical document to search your vector store.

➡️ **HyDE bridges the gap between user intent and relevant content, especially when:**

1. Queries are short.
2. There is a language mismatch between the query and documents.
3. You want to retrieve based on **answer content**, not just **question words**.

### Manual HyDE

In [2]:
import os

os.environ["USER_AGENT"] = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/137.0.0.0 Safari/537.36"
)

In [5]:
from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.vectorstores import Chroma

In [6]:
# 1. Load and chunk your dataset

chunk_size = 300
chunk_overlap =100

# loading data
loader = WikipediaLoader(query="Steve Jobs", load_max_docs=5)
documents =loader.load()

# text splitting
text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size,chunk_overlap=chunk_overlap)
docs = text_splitter.split_documents(documents=documents)
docs

[Document(metadata={'title': 'Steve Jobs', 'summary': 'Steven Paul Jobs  (February 24, 1955 – October 5, 2011) was an American businessman, inventor, and investor. A pioneer of the personal computer revolution of the 1970s and 1980s, Jobs co-founded Apple Inc. with his early business partner Steve Wozniak as Apple Computer Company in 1976. After the company\'s board of directors fired him in 1985, he founded NeXT the same year and purchased Pixar in 1986, becoming its chairman and majority shareholder until 2007. Jobs returned to Apple in 1997 as CEO, where he was closely involved with the creation and promotion of many of the company\'s most influential products until his resignation in 2011.\nJobs was born in San Francisco in 1955 and adopted shortly afterward. He attended Reed College in 1972 before withdrawing that same year. In 1974, he traveled through India, seeking enlightenment before later studying Zen Buddhism. He and Wozniak co-founded Apple in 1976 to further develop and s

In [7]:
from langchain_classic.vectorstores import FAISS
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorStore = FAISS.from_documents(docs,embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6108.70it/s]


In [8]:


import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

from langchain.chat_models import init_chat_model

llm = init_chat_model(
    "llama-3.3-70b-versatile",
    model_provider="groq"
)
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000251F2E02DE0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000251F2E03BF0>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [9]:
from langchain_classic.vectorstores import Chroma

# creating vectorStore
db = Chroma.from_documents(documents=docs,embedding=embeddings,persist_directory="output/Steve_jobs_for_HyDE.db")
## Create the retriever
base_retriever =db.as_retriever(search_kwargs = {"k":5})


In [10]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
)
## Generating a prompt for generating HyDE

def get_hypo_doc(query):
    template = """Imagine You are an expert writing a detailed explanation on the topic: '{query}'
    create a hypothetical answer for the topic."""
    
    system_message_prompt = SystemMessagePromptTemplate.from_template(template=template)
    chat_prompt = ChatPromptTemplate.from_messages([system_message_prompt])
    messages = chat_prompt.format_prompt(query=query).to_messages()
    print(messages)
    response = llm.invoke(messages)
    hypo_doc = response.content
    return hypo_doc
    

In [11]:
query = 'When was Steve Jobs Fired From Apple?'
print(get_hypo_doc(query=query))

[SystemMessage(content="Imagine You are an expert writing a detailed explanation on the topic: 'When was Steve Jobs Fired From Apple?'\n    create a hypothetical answer for the topic.", additional_kwargs={}, response_metadata={})]
**The Turbulent Era: When Steve Jobs Was Fired From Apple**

It's a well-known fact that Steve Jobs, the visionary co-founder of Apple, was indeed fired from the company he helped create. This pivotal event occurred in 1985, marking a significant turning point in Jobs' career and the future of Apple.

**The Background:**

In the early 1980s, Apple was experiencing a period of rapid growth, thanks in large part to the success of the Apple II computer. However, the company's board of directors, led by CEO John Sculley, began to clash with Jobs over the direction of the company. Jobs, who had been instrumental in developing the Macintosh computer, had a clear vision for Apple's future, but his strong personality and tendency to micromanage often put him at odds 

In [12]:
matches_doc = base_retriever.invoke(get_hypo_doc(query))
print(matches_doc)

[SystemMessage(content="Imagine You are an expert writing a detailed explanation on the topic: 'When was Steve Jobs Fired From Apple?'\n    create a hypothetical answer for the topic.", additional_kwargs={}, response_metadata={})]
[Document(metadata={'source': 'https://en.wikipedia.org/wiki/Steve_Jobs', 'title': 'Steve Jobs', 'summary': 'Steven Paul Jobs  (February 24, 1955 – October 5, 2011) was an American businessman, inventor, and investor. A pioneer of the personal computer revolution of the 1970s and 1980s, Jobs co-founded Apple Inc. with his early business partner Steve Wozniak as Apple Computer Company in 1976. After the company\'s board of directors fired him in 1985, he founded NeXT the same year and purchased Pixar in 1986, becoming its chairman and majority shareholder until 2007. Jobs returned to Apple in 1997 as CEO, where he was closely involved with the creation and promotion of many of the company\'s most influential products until his resignation in 2011.\nJobs was bo

In [ ]:
#Create RAG Prompt
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_template(
"""
You are an AI assistant.

Use ONLY the context below to answer the user's question.

If the answer is not available in the context, simply say
"I don't have enough information."

------------------------
Context:
{context}
------------------------

Question:
{question}

Answer:
"""
)

In [14]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()


def hyde_rag_pipeline(query):

    # Step 1 : Generate Hypothetical Document
    hypothetical_doc = get_hypo_doc(query)

    # Step 2 : Retrieve Documents using HyDE
    retrieved_docs = base_retriever.invoke(hypothetical_doc)

    # Step 3 : Convert Documents into Context
    context = "\n\n".join(
        [doc.page_content for doc in retrieved_docs]
    )

    # Step 4 : Create Prompt
    prompt = rag_prompt.invoke({
        "context": context,
        "question": query
    })

    # Step 5 : Generate Final Answer
    response = llm.invoke(prompt)

    # Step 6 : Parse Output
    final_answer = output_parser.invoke(response)

    return {
        "query": query,
        "hypothetical_document": hypothetical_doc,
        "retrieved_docs": retrieved_docs,
        "context": context,
        "answer": final_answer
    }

In [15]:
query = "When was Steve Jobs fired from Apple?"

result = hyde_rag_pipeline(query)

[SystemMessage(content="Imagine You are an expert writing a detailed explanation on the topic: 'When was Steve Jobs fired from Apple?'\n    create a hypothetical answer for the topic.", additional_kwargs={}, response_metadata={})]


In [16]:
print("Question:")
print(result["query"])

print("\n" + "="*80)

print("Hypothetical Document:")
print(result["hypothetical_document"])

print("\n" + "="*80)

print("Retrieved Documents:")

for i, doc in enumerate(result["retrieved_docs"], 1):
    print(f"\nDocument {i}:")
    print(doc.page_content)

print("\n" + "="*80)

print("Context:")
print(result["context"])

print("\n" + "="*80)

print("Final Answer:")
print(result["answer"])

Question:
When was Steve Jobs fired from Apple?

Hypothetical Document:
**The Turbulent Era: Understanding the Circumstances Surrounding Steve Jobs' Departure from Apple**

Steve Jobs, the visionary co-founder of Apple Inc., was indeed fired from the company he helped create. This pivotal event occurred in 1985, marking a significant turning point in both Jobs' career and Apple's history.

**Background: The Rise of Steve Jobs and Apple**

To comprehend the context of Jobs' departure, it's essential to delve into the early days of Apple. Founded in 1976 by Jobs, Steve Wozniak, and Ronald Wayne, Apple quickly gained recognition for its innovative personal computers, particularly the Apple II. The company's success was largely attributed to the duo's passion for design and technology. However, as Apple expanded, the need for professional management became apparent.

**The Power Struggle: Jobs vs. Sculley**

In 1980, Apple introduced the Apple III, which, despite its initial hype, failed t

### Custom HyDE

@Langchain - HypotheticalDocumentEmbedder

In [28]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.hyde.base import HypotheticalDocumentEmbedder



In [52]:
# Step 1: Load the dataset and split the data

loader = TextLoader("langchain_crewai_dataset.txt")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(docs)

In [53]:
# embedding and llm 
base_embeddings= HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

from langchain.chat_models import init_chat_model

llm = init_chat_model(
    "llama-3.3-70b-versatile",
    model_provider="groq"
)
llm

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9272.26it/s]


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000025234B86F00>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000251F46B1880>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

According to the official documentation and LangChain source code (mapping in `PROMPT_MAP`), the default options are:

- `web_search`
- `sci_fact`
- `arguana`
- `trec_covid`
- `fiqa`
- `dbpedia_entity`
- `trec_news`
- `mr_tydi`

In [54]:
hyde_embedding_function = HypotheticalDocumentEmbedder.from_llm(
    llm=llm,
    base_embeddings = base_embeddings,
    prompt_key = "web_search"
)

In [55]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=hyde_embedding_function,
    persist_directory="Output/steve_job_hyde_chains"
)

In [56]:
# Step 5: RAG answer generation prompt

rag_prompt = PromptTemplate.from_template("""
Use the context below to answer the question.

Context:
{context}

Question: {input}
""")

rag_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=rag_prompt
)

In [59]:
# Step 6: Final RAG Pipeline

def hyde_rag_pipeline(query):
    matched_docs = vectorstore.similarity_search(query, k=4)
    print(matched_docs)

    response = rag_chain.invoke({
        "input": query,
        "context": matched_docs
    })

    return response

In [61]:
# Step 7: Run example query

query = "What memory modules does LangChain provide?"

answer = hyde_rag_pipeline(query)

print("✅ Final Answer:\n", answer)

[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain offers memory modules like ConversationBufferMemory and ConversationSummaryMemory. These allow the LLM to maintain awareness of previous conversation turns or summarize long interactions to fit within token limits. (v10)'), Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain offers memory modules like ConversationBufferMemory and ConversationSummaryMemory. These allow the LLM to maintain awareness of previous conversation turns or summarize long interactions to fit within token limits. (v10)'), Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain offers memory modules like ConversationBufferMemory and ConversationSummaryMemory. These allow the LLM to maintain awareness of previous conversation turns or summarize long interactions to fit within token limits. (v10)'), Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_co

### Custom Prompt 



In [62]:
from langchain_core.prompts import PromptTemplate
custom = PromptTemplate.from_template(
    "Generate a concise hypothetical answer for this topic: {query}"
)

# Step 3: HyDE Embedder using custom prompt
hyde_embedding_function = HypotheticalDocumentEmbedder.from_llm(
    llm=llm,
    base_embeddings=base_embeddings,
    custom_prompt=custom
)

# 🎯 Why Use HyDE?

| **Problem** | **How HyDE Helps** |
|--------------|--------------------|
| User query and document use different words | Embeds answer-style content instead of question |
| Query is too vague | LLM-generated answer gives richer semantics |
| Better grounding needed | Embeds what the answer might look like |
| Zero-shot retrieval | Works well even without prior training |

# ✅ Benefits of HyDE

| **Feature** | **Why It Helps** |
|-------------|------------------|
| Semantic intent modeling | Better than literal keyword matching |
| LLM-aware retrieval | Query is expanded into more context |
| Generalization | Works well even if document phrasing differs from question |
| Plug-and-play | No need for retraining; works with OpenAI, Cohere, Hugging Face |